In [1]:
import time, math, osmnx as ox

t0 = time.time()
G = ox.graph_from_point((43.22, 76.90), dist=3000, network_type="drive") 
print(f"{time.time()-t0:.1f}s, {len(G.nodes)} nodes, {len(G.edges)} edges")

2.6s, 1650 nodes, 3913 edges


Graph around central Almaty (3 km radius): **1650 nodes, 3913 edges**, loaded in ~2.5 s from the OSMnx cache. A fresh download from Overpass takes longer and can fail when the server is overloaded, which matters for Module 1's latency benchmark: build graphs offline, never per request.

In [2]:
def haversine(lon1, lat1, lon2, lat2, R=6_371_008.8):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(h))

# Main Botanical Garden: north side (Timiryazev St) vs south side.
# Sairan needs a new graph download, and Overpass is unreachable right now,
# so this pair uses the cached graph from the first cell.
orig_lon, orig_lat = 76.9100, 43.2265
dest_lon, dest_lat = 76.9100, 43.2150

orig, d_o = ox.distance.nearest_nodes(G, X=orig_lon, Y=orig_lat, return_dist=True)
dest, d_d = ox.distance.nearest_nodes(G, X=dest_lon, Y=dest_lat, return_dist=True)
print(f"snap: {d_o:.0f} m, {d_d:.0f} m")
assert orig != dest, "both points snapped to the same node"

route = ox.routing.shortest_path(G, orig, dest, weight="length")
road_m = ox.routing.route_to_gdf(G, route)["length"].sum()
o, d = G.nodes[orig], G.nodes[dest]
straight_m = haversine(o["x"], o["y"], d["x"], d["y"])
print(f"road {road_m:.0f} m, straight {straight_m:.0f} m, detour {road_m/straight_m:.2f}")

snap: 47 m, 14 m
road 2520 m, straight 1230 m, detour 2.05


In [3]:
import folium

# GeoDataFrame.explore() needs matplotlib + mapclassify, so draw with plain folium.
# Edge geometries keep the real road shape; folium wants (lat, lon), shapely gives (lon, lat).
route_gdf = ox.routing.route_to_gdf(G, route)
m = folium.Map(location=((o["y"] + d["y"]) / 2, (o["x"] + d["x"]) / 2), zoom_start=15)
for geom in route_gdf.geometry:
    folium.PolyLine([(y, x) for x, y in geom.coords], color="red", weight=5).add_to(m)
folium.PolyLine([(o["y"], o["x"]), (d["y"], d["x"])],
                color="blue", weight=3, dash_array="8").add_to(m)
folium.Marker((o["y"], o["x"]), tooltip="A: Timiryazev St").add_to(m)
folium.Marker((d["y"], d["x"]), tooltip="B: south of the garden").add_to(m)

m.save("../detour_map.html")   # m00_environment/detour_map.html
m

## Result: detour index 2.05

| | Distance |
|---|---|
| Straight line (haversine, between the snapped nodes) | 1230 m |
| Shortest drive route | 2520 m |
| **Detour index** | **2.05** |

**Why:** the Main Botanical Garden is a large closed block with no through roads, so a car on Timiryazev St has to drive around it (via Zharokov and Utepov streets) to reach the other side. The map (`../detour_map.html`) shows the red route against the blue straight line.

**Why haversine ranking fails here:** a driver at point B looks 1.2 km away from a rider at A, but the real drive is 2.5 km, so a driver 2 km away on the same street would arrive sooner and be ranked lower. This is the "nearest driver is 8 minutes away" bug from the DeepETA post: straight-line distance is a candidate filter, not an ETA.

*Note:* the planned Sairan lake pair needs a new graph download; Overpass was unreachable on 2026-10-14, so this pair uses the cached graph.